# 09 — ModelRuntime and Provider-Neutral Execution

**Mode:** Offline  
**Level:** Advanced  
**Estimated time:** 35 minutes

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A provider-neutral execution boundary using a deterministic local adapter. You will inspect the normalized request, provider/model selection, declared capabilities, sync and async responses, native sync and async streams, usage, reasoning controls, and structured output.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(9, 'ModelRuntime and Provider-Neutral Execution')


## Prerequisites and setup

**Course prerequisites:** Complete `course-01-hello-world`, `course-05-tool-use`.

**Execution requirements:** Offline. The local adapter is infrastructure for a repeatable lesson. Protected live certification exercises the same ModelRuntime path with every supported provider family.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Build and inspect a provider-neutral ModelRequest.
- Resolve provider/model capabilities before execution.
- Use sync and async invocation through one runtime.
- Consume normalized sync/async stream events, usage, and structured output.


## Mental model

Agent code should not assemble five different provider payloads. **ModelRuntime** accepts provider-neutral messages and options, checks them against provider/model capabilities, calls the selected adapter, and returns normalized ModelResponse or ModelEvent objects. Provider-specific code stays behind that boundary.


In [ ]:
show_flow(
('Agent', 'neutral request', 'agent'),
('ModelRuntime', 'validate and normalize', 'reef'),
('Provider adapter', 'native API shape', 'provider'),
('Response', 'neutral events and usage', 'spore'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Define a transparent local adapter

The adapter records each real ModelRequest it receives. It declares only the capabilities used in this lesson and returns normalized model objects.


In [ ]:
import json

from praval.models import (
    ModelEvent, ModelResponse, ProviderCapabilities, Usage,
)


class CourseRuntimeProvider:
    provider_name = "course-runtime"
    capabilities = ProviderCapabilities(
        text=True, streaming=True, native_streaming=True,
        structured_outputs=True, reasoning=True, reasoning_effort=True,
    )

    def __init__(self):
        self.requests = []

    def response(self, request):
        self.requests.append(request)
        stage("provider", "invoke", request.model)
        return ModelResponse(
            content=json.dumps({"summary": "runtime contracts are explicit"}),
            provider=self.provider_name, model=request.model,
            usage=Usage(input_tokens=4, output_tokens=3, total_tokens=7),
            finish_reason="stop",
        )

    def invoke(self, request, tools=None):
        return self.response(request)

    async def ainvoke(self, request, tools=None):
        return self.response(request)


### What just happened?

The adapter has no credential or network behavior, but it implements the same contract a live provider adapter implements.

### Why this matters

A transparent adapter lets this lesson inspect runtime normalization instead of teaching one vendor SDK.


### Add native streaming methods

Sync and async streams emit the same provider-neutral event types. A final event contains the assembled response and usage.


In [ ]:
def sync_stream(self, request, tools=None):
    for delta in ("Praval ", "streams"):
        yield ModelEvent(type="delta", delta=delta)
    usage = Usage(input_tokens=2, output_tokens=2, total_tokens=4)
    yield ModelEvent(type="usage", usage=usage)
    yield ModelEvent(
        type="final",
        response=ModelResponse(
            content="Praval streams", provider=self.provider_name,
            model=request.model, usage=usage,
        ),
        usage=usage,
    )


async def async_stream(self, request, tools=None):
    for event in sync_stream(self, request, tools):
        yield event


CourseRuntimeProvider.stream = sync_stream
CourseRuntimeProvider.astream = async_stream


### What just happened?

The adapter now fulfills its `native_streaming=True` declaration for both calling styles.

### Why this matters

Capabilities are promises. An adapter should not advertise behavior it does not implement.


### Construct the runtime and inspect capabilities

Provider selection and model selection are explicit constructor inputs. Agent configuration supplies request defaults without leaking provider SDK objects.


In [ ]:
from praval.core.agent import AgentConfig
from praval.model_runtime import ModelRuntime

provider = CourseRuntimeProvider()
runtime = ModelRuntime(
    provider=provider,
    provider_name="course-runtime",
    config=AgentConfig(
        provider="course-runtime", model="course-model",
        temperature=0, max_output_tokens=80, retries=1,
    ),
)
capabilities = runtime.capabilities
assert capabilities.streaming and capabilities.structured_outputs
assert capabilities.reasoning_effort
show_json(capabilities.model_dump(), "Declared provider capabilities")


### What just happened?

The runtime now knows both what to call and which features may enter a request.

### Why this matters

Capability checks turn unsupported combinations into early framework errors instead of unpredictable provider responses.


### Make a structured sync request

The JSON schema and reasoning effort are neutral controls. The adapter records their normalized representation.


In [ ]:
schema = {
    "type": "object",
    "properties": {"summary": {"type": "string"}},
    "required": ["summary"],
    "additionalProperties": False,
}
response = runtime.invoke(
    messages=[{"role": "user", "content": "Explain the runtime."}],
    response_schema=schema,
    reasoning={"effort": "low"},
    metadata={"lesson": 9},
)
payload = json.loads(response.content)
assert payload["summary"] == "runtime contracts are explicit"
assert response.usage.total_tokens == 7
stage("runtime", "sync response", response.content)


### What just happened?

The result has normalized content, provider, model, finish reason, and Usage. The captured request has typed schema and reasoning objects.

### Why this matters

Structured output is only useful when the application validates the resulting shape; provider support alone is not a business assertion.


In [ ]:
captured = provider.requests[-1]
show_json(
    {
        "request": captured.model_dump(exclude_none=True),
        "response": response.model_dump(exclude_none=True),
        "parsed": payload,
    },
    "Normalized sync exchange",
)


### Use the async invocation path

`ainvoke()` keeps asynchronous providers and tools on the caller's event loop while preserving the same request and response models.


In [ ]:
async_response = await runtime.ainvoke(
    messages=[{"role": "user", "content": "Run asynchronously."}],
    response_schema=schema,
    reasoning={"effort": "low"},
)
assert json.loads(async_response.content) == payload
assert async_response.usage.total_tokens == 7
stage("runtime", "async response", async_response.content)
show_json(async_response.model_dump(exclude_none=True), "Normalized async response")


### What just happened?

The asynchronous adapter method executed without moving the work to another event loop or changing the public response type.

### Why this matters

One contract lets applications choose concurrency style independently from provider shape.


### Consume sync and async streams

Every stream begins with runtime metadata, carries deltas and usage, and ends with a final response. Applications should not infer completion from a quiet socket.


In [ ]:
sync_events = list(runtime.stream(
    messages=[{"role": "user", "content": "Stream a short answer."}]
))
async_events = []
async for event in runtime.astream(
    messages=[{"role": "user", "content": "Stream asynchronously."}]
):
    async_events.append(event)

for label, events in (("sync", sync_events), ("async", async_events)):
    deltas = [event.delta for event in events if event.type == "delta"]
    assert "".join(deltas) == "Praval streams"
    assert events[0].type == "start" and events[-1].type == "final"
    assert any(event.type == "usage" for event in events)
    for event in events:
        stage("runtime", f"{label}:{event.type}", event.delta or "")

show_json(
    [event.model_dump(exclude_none=True) for event in async_events],
    "Normalized async stream",
)
show_timeline()


### What just happened?

Both paths exposed the same semantic event sequence even though one used an iterator and the other an async iterator.

### Why this matters

Normal event types keep user interfaces and instrumentation provider-neutral.


## Your turn

Add a `request_id` to request metadata and have the adapter copy it into response metadata. Inspect the round trip for both `invoke` and `ainvoke`.


In [ ]:
# In CourseRuntimeProvider.response(), read request.metadata["request_id"].
# Return it in ModelResponse(metadata={"request_id": ...}).
# Assert both sync and async callers receive the same value.


## Common mistake

**Mistake:** Sending provider API keys or authorization headers through `provider_options`.

**Correction:** Configure credentials on provider construction or environment boundaries. ModelRuntime rejects credential-bearing request options.


<details>
<summary>Under the hood</summary>

ModelRuntime builds a typed ModelRequest, merges safe profile defaults, resolves capabilities, validates feature combinations, applies bounded retries, invokes the adapter, orchestrates tools when present, and normalizes the response or stream.

</details>


## Recap

- Provider-neutral requests isolate agents from SDK payloads.
- Capabilities are validated promises tied to provider and model.
- Sync and async paths return the same semantic response objects.
- Streams have explicit start, delta, usage, and final events.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
provider.requests.clear()
show_callout("Runtime released", "The local adapter holds no external resources.", role="provider")


### Next lesson

Continue to `10_human_in_the_loop.ipynb` to pause a real model-generated tool call, persist it, apply human decisions, and resume safely.
